# Dashboard Interaktif - Auto MPG Dataset

Nama  : Ikhsan Maulana

## Install dan Import Library

In [ ]:
# Install library
!pip install panel hvplot bokeh pandas -q

In [ ]:
# Import library
import pandas as pd
import panel as pn
import hvplot.pandas
import numpy as np


pn.extension('tabulator', design='material')

## Bersihkan Cache Untuk Performa

In [ ]:
# Cache Data untuk Performa
if 'data' not in pn.state.cache.keys():
    df = pd.read_csv('auto-mpg.csv')
    pn.state.cache['data'] = df.copy()
else:
    df = pn.state.cache['data']


In [ ]:
# Preview data utama
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


## Preprocessing Data Sederhana

In [ ]:
# Preprocessing Minimal
# Ganti '?' di horsepower dengan NaN lalu ubah ke numerik
df['horsepower'] = pd.to_numeric(df['horsepower'].replace('?', np.nan), errors='coerce')

# Isi nilai kosong horsepower dengan median
df['horsepower'] = df['horsepower'].fillna(df['horsepower'].median())

# Pastikan kolom numerik benar
num_cols = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

In [ ]:
# Ubah kode origin jadi label
origin_map = {1: 'USA', 2: 'Europe', 3: 'Japan'}
df['origin'] = df['origin'].map(origin_map)


## Membuat Widget

In [ ]:
# Widget Utama
year_slider_global = pn.widgets.IntSlider(
    name='Tahun Produksi (Global)', start=int(df['model year'].min()), end=int(df['model year'].max()), value=76)
origin_select_global = pn.widgets.Select(
    name='Asal Mobil (Global)', options=['All'] + sorted(df['origin'].dropna().unique().tolist()))

In [ ]:
# Widget khusus untuk tiap visualisasi
# Untuk Tren Rata-rata MPG per Tahun
year_slider_trend = pn.widgets.IntRangeSlider(
    name='Rentang Tahun (Tren MPG)', start=int(df['model year'].min()), end=int(df['model year'].max()),
    value=(70, 82))

# 2️⃣ Untuk Rata-rata Berdasarkan Silinder
year_select_cyl = pn.widgets.IntSlider(
    name='Tahun (MPG vs Silinder)', start=int(df['model year'].min()), end=int(df['model year'].max()), value=76)

# 3️⃣ Untuk Distribusi MPG Berdasarkan Asal
origin_select_dist = pn.widgets.Select(
    name='Asal Mobil (Distribusi MPG)', options=['All'] + sorted(df['origin'].dropna().unique().tolist()))

## Membuat Visualisasi Sebagai Jawaban dari Pertanyaan Analisis

In [ ]:
# Pertanyaan 1: Bagaimana tren rata-rata mpg dari tahun ke tahun
@pn.depends(year_slider_trend.param.value, origin_select_global.param.value)
def plot_avg_mpg_trend(year_range, origin):
    start_year, end_year = year_range
    dff = df[
        (df['model year'].between(start_year, end_year)) &
        ((df['origin'] == origin) | (origin == 'All'))
    ]
    avg_mpg = dff.groupby('model year')['mpg'].mean().reset_index()
    return avg_mpg.hvplot.line(
        x='model year', y='mpg', line_width=3, color='#007BFF',
        title='Tren Rata-rata MPG per Tahun'
    )

In [ ]:
# Pertanyaan 2: Apakah ada perbedaan rata-rata mpg berdasarkan cylinders
@pn.depends(year_select_cyl.param.value, origin_select_global.param.value)
def plot_avg_mpg_by_cyl(year, origin):
    dff = df[
        (df['model year'] == year) &
        ((df['origin'] == origin) | (origin == 'All'))
    ]
    avg_mpg = dff.groupby('cylinders')['mpg'].mean().reset_index()
    return avg_mpg.hvplot.bar(
        x='cylinders', y='mpg', color='#007BFF',
        title=f'Rata-rata MPG Berdasarkan Jumlah Silinder (Tahun {year})'
    )

In [ ]:
# Pertanyaan 3: Bagaimana distribusi nilai mpg pada seluruh mobil
@pn.depends(year_slider_global.param.value, origin_select_dist.param.value)
def plot_mpg_distribution(year, origin):
    dff = df[
        (df['model year'] <= year) &
        ((df['origin'] == origin) | (origin == 'All'))
    ]
    return dff.hvplot.hist(
        y='mpg', bins=20, color='#007BFF',
        title=f'Distribusi Nilai MPG ({origin}) hingga Tahun {year}'
    )

## Membuat Dashboard

In [ ]:
import panel as pn

# Aktifkan tema light + custom CSS
pn.extension(
    'tabulator',
    theme='default',
    raw_css=[ """
    /* === Styling untuk MODE TERANG (default theme) === */
    .bk.panel-default-theme .bk.panel-markdown,
    .bk.panel-default-theme .bk.panel-title,
    .bk.panel-default-theme .bk.card-title,
    .bk.panel-default-theme .bk.panel-side {
        color: #0D47A1 !important; /* biru tua */
    }

    /* Warna teks biasa */
    .bk.panel-default-theme .bk.panel-markdown p,
    .bk.panel-default-theme .bk.panel-markdown li {
        color: #212121 !important; /* abu tua */
    }

    /* Warna judul sidebar */
    .bk.panel-default-theme .bk.panel-markdown h2,
    .bk.panel-default-theme .bk.panel-markdown h3 {
        color: #0D47A1 !important;
    }

    /* === Styling untuk MODE GELAP (dark theme) === */
    .bk.panel-dark-theme .bk.panel-markdown,
    .bk.panel-dark-theme .bk.panel-title,
    .bk.panel-dark-theme .bk.card-title,
    .bk.panel-dark-theme .bk.panel-side {
        color: #ECEFF1 !important; /* abu terang */
    }
    """ ]
)

template = pn.template.FastListTemplate(
    title='Dashboard Analisis Auto MPG',
    sidebar=[
        pn.pane.Markdown("## Tentang Dataset"),
        pn.pane.Markdown("""
        Dataset ini berisi informasi tentang **efisiensi bahan bakar (MPG)**,
        **jumlah silinder**, **tenaga mesin**, dan **asal mobil**.
        """),
        pn.pane.Markdown("## 🔧 Pengaturan Global"),
        year_slider_global,
        origin_select_global,
    ],
    main=[
        pn.Card(plot_avg_mpg_trend, year_slider_trend, title="Tren Rata-rata MPG per Tahun", collapsed=False),
        pn.Card(plot_avg_mpg_by_cyl, year_select_cyl, title="Rata-rata MPG Berdasarkan Silinder", collapsed=False),
        pn.Card(plot_mpg_distribution, origin_select_dist, title="Distribusi Nilai MPG Berdasarkan Seluruh Mobil", collapsed=False),
    ],
    accent_base_color="#007BFF",
    header_background="#007BFF",
    theme="default"
)

template.servable()
pn.panel(template)

FastListTemplate
    [js_area] HTML(None, design=<class 'panel.theme.materi..., height=0, margin=0, sizing_mode='fixed', width=0)
    [actions] TemplateActions()
    [browser_info] BrowserInfo()
    [busy_indicator] LoadingSpinner(height=20, width=20)
    [main-1660164517872] Card(design=<class 'panel.theme.materi..., title='📈 Tren Rata-rata M...)
        [0] ParamFunction(function, _pane=HoloViews, defer_load=False, design=<class 'panel.theme.materi...)
        [1] IntRangeSlider(design=<class 'panel.theme.materi..., end=82, name='Rentang Tahun (..., start=70, value=(70, 82), value_end=82, value_start=70)
    [main-1660164509472] Card(design=<class 'panel.theme.materi..., title='⚙️ Rata-rata M...)
        [0] ParamFunction(function, _pane=HoloViews, defer_load=False, design=<class 'panel.theme.materi...)
        [1] IntSlider(design=<class 'panel.theme.materi..., end=82, name='Tahun (MPG vs Silinder)', start=70, value=76)
    [main-1660293772928] Card(design=<class 'panel.theme.materi..., title='📊 Distribusi N...)
        [0] ParamFunction(function, _pane=HoloViews, defer_load=False, design=<class 'panel.theme.materi...)
        [1] Select(design=<class 'panel.theme.materi..., name='Asal Mobil (Distribusi M..., options=['All', 'Europe', ...], value='All')
    [nav-1660269016032] Markdown(str, design=<class 'panel.theme.materi...)
    [nav-1660291453568] Markdown(str, design=<class 'panel.theme.materi...)
    [nav-1660179764272] Markdown(str, design=<class 'panel.theme.materi...)
    [nav-1660291771392] IntSlider(design=<class 'panel.theme.materi..., end=82, name='Tahun Produksi (Global)', start=70, value=76)
    [nav-1660291452752] Select(design=<class 'panel.theme.materi..., name='Asal Mobil (Global)', options=['All', 'Europe', ...], value='All')

## Narasi Insight dan Pertanyaan Analisis
1. Tren Rata-rata MPG per Tahun

  - Pertanyaan Analisis: Bagaimana perkembangan efisiensi bahan bakar (MPG) mobil dari tahun ke tahun?

  - Insight Narasi:
  - Dari grafik tren rata-rata MPG per tahun, terlihat adanya peningkatan efisiensi bahan bakar seiring waktu. Mobil-mobil keluaran tahun 70-an awal umumnya memiliki nilai MPG rendah, menandakan konsumsi bahan bakar yang boros. Namun, memasuki akhir 70-an hingga 80-an, rata-rata MPG meningkat signifikan.
  - Hal ini kemungkinan dipengaruhi oleh krisis minyak dan kebijakan efisiensi energi, yang mendorong produsen mobil mengembangkan mesin lebih hemat bahan bakar. Artinya, secara umum, teknologi otomotif semakin efisien dari waktu ke waktu.
  <br>

2. Rata-rata MPG Berdasarkan Jumlah Silinder

  - Pertanyaan Analisis: Apakah jumlah silinder mesin berpengaruh terhadap efisiensi bahan bakar mobil?

  - Insight Narasi:
  - Grafik rata-rata MPG berdasarkan jumlah silinder menunjukkan hubungan terbalik antara jumlah silinder dan efisiensi bahan bakar. Mobil dengan 4 silinder memiliki rata-rata MPG tertinggi, sedangkan mobil 6 atau 8 silinder menunjukkan nilai yang jauh lebih rendah.
  - Hal ini masuk akal karena semakin banyak silinder, kapasitas mesin dan konsumsi bahan bakar meningkat, sehingga efisiensinya menurun. Kesimpulannya, mobil dengan silinder lebih sedikit cenderung lebih hemat bahan bakar, sementara mobil bersilinder besar biasanya berfokus pada performa, bukan efisiensi.
  <BR>

3. Distribusi Nilai MPG Berdasarkan Asal Mobil

  - Pertanyaan Analisis: Bagaimana perbandingan efisiensi bahan bakar mobil berdasarkan asal (region of origin)?

  - Insight Narasi:
  - Distribusi MPG berdasarkan asal mobil memperlihatkan bahwa mobil dari Jepang dan Eropa cenderung memiliki nilai MPG lebih tinggi dibanding mobil dari Amerika.
  - Mobil Amerika biasanya berukuran besar dengan tenaga besar, sehingga konsumsi bahan bakarnya tinggi. Sebaliknya, produsen Jepang dan Eropa dikenal dengan desain kompak dan efisien, yang membuat nilai MPG mereka lebih baik.
  - Dengan demikian, terlihat adanya perbedaan karakter industri otomotif antar wilayah, di mana efisiensi menjadi fokus utama bagi produsen Asia dan Eropa.